In [ ]:
# pip install llama-index-llms-openai llama-index-vector-stores-chroma


In [1]:

# pip install llama-index-embeddings-openai 

In [1]:
import os
import chromadb
from dotenv import load_dotenv

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    StorageContext,
)
from llama_index.core.node_parser import SentenceSplitter
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore


In [2]:
load_dotenv()
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0)



/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
docs = SimpleDirectoryReader(
    input_files=[
        "data_test/hr_policy_manual.pdf",
        "data_test/novacrm_product_manual.pdf",
        "data_test/annual_financial_report_2024.pdf",
    ]
).load_data()

In [4]:
nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(docs)


In [6]:
chroma_client = chromadb.PersistentClient(path="./nb03_chroma")


col = chroma_client.create_collection("docs_v1")
vs = ChromaVectorStore(chroma_collection=col)
sc = StorageContext.from_defaults(vector_store=vs)
VectorStoreIndex(nodes, storage_context=sc)

2026-06-07 12:24:33,143 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [8]:

vs

ChromaVectorStore(stores_text=True, is_embedding_query=True, flat_metadata=True, collection_name=None, host=None, port=None, ssl=False, headers=None, persist_dir=None, collection_kwargs={})

In [9]:
index = VectorStoreIndex.from_vector_store(vs, storage_context=sc)

In [10]:
query = "What is the WFH policy?"
retriever_basic = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever_basic.retrieve(query)


2026-06-07 12:26:09,257 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [11]:
retrieved_nodes

[NodeWithScore(node=TextNode(id_='c6458b31-1ebe-4cc8-b703-1e7a3863ebe1', embedding=None, metadata={'page_label': '1', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a55c79be-8269-4124-8f08-e4dc0ada1767', node_type='4', metadata={'page_label': '1', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, hash='19b98eb08572b2fb01055d502848db2264094d44f6d472da07523a0

In [ ]:
# metadatafiltering
from llama_index.core.vector_stores.types import MetadataFilters, ExactMatchFilter

query = "What is the leave policy?"

# 1. Create a filter that allows only chunks from the HR policy PDF
hr_filter = MetadataFilters(
    filters=[
        ExactMatchFilter(key="file_name", value="hr_policy_manual.pdf")
    ]
)



retriever_filtered = index.as_retriever(
    similarity_top_k=3,
    filters=hr_filter
)



retrieved_nodes = retriever_filtered.retrieve(query)
print(retrieved_nodes)


2026-06-07 12:28:08,002 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[NodeWithScore(node=TextNode(id_='29aca3be-a468-4199-9f12-8b32c25fcc2d', embedding=None, metadata={'page_label': '2', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='0a0c6b86-dc8b-42be-90f1-ad6c650487ff', node_type='4', metadata={'page_label': '2', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, hash='dd4ae9304b45a936eb1ca231998de749f43c24653cfe2485717625e

In [18]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever


In [16]:
engine = RetrieverQueryEngine(retriever=retriever_filtered)

In [17]:
engine.query("hi")

2026-06-07 12:32:29,650 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 12:32:30,782 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response(response="I'm here to help. How can I assist you today?", source_nodes=[NodeWithScore(node=TextNode(id_='60f25f41-fab8-4e19-aac8-636cc78357f6', embedding=None, metadata={'page_label': '3', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='daecbf8c-cfa9-409b-b63f-1da410077632', node_type='4', metadata={'page_label': '3', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_dat

In [19]:
# col = chroma_client.create_collection("docs_v1")
# vs = ChromaVectorStore(chroma_collection=col)

In [ ]:
# sc = StorageContext.from_defaults(vector_store=vs)




index = VectorStoreIndex(nodes)

2026-06-06 14:02:07,960 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [13]:
query_engine = index.as_query_engine()  

response = query_engine.query("What is the HR Policy Manual?")
print(response)

2026-06-06 14:02:45,576 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-06 14:02:46,856 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The HR Policy Manual is a document that outlines the employment terms, leave policies, and other guidelines for employees at NovaTech Solutions Pvt. Ltd. It includes information on probation periods, notice periods, working hours, annual leave, sick leave, and maternity leave policies.


In [15]:
retriever_basic = index.as_retriever(similarity_top_k=3)

response = retriever_basic.retrieve("What is the HR Policy Manual?")
print(response)

2026-06-06 14:04:31,666 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


[NodeWithScore(node=TextNode(id_='ab4a953e-49b1-4751-90ab-d0a406b420fd', embedding=None, metadata={'page_label': '4', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8ee787e0-4130-4bce-9f19-a69f411ef152', node_type='4', metadata={'page_label': '4', 'file_name': 'hr_policy_manual.pdf', 'file_path': 'data_test/hr_policy_manual.pdf', 'file_type': 'application/pdf', 'file_size': 7583, 'creation_date': '2026-06-01', 'last_modified_date': '2026-06-06'}, hash='9ea0402d21602a54658227aeb17cee36b4d4cb6f8255984302a5a94

In [16]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(
    retriever=retriever_basic

)

In [17]:
query = "what is the HR Policy Manual?"

response = query_engine.query(query)
print(response)

2026-06-06 14:06:07,660 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-06 14:06:08,803 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The HR Policy Manual is a document that outlines the various policies and guidelines related to employment terms, leave policies, compensation and benefits, performance appraisal cycles, code of conduct, and disciplinary actions at NovaTech Solutions Pvt. Ltd.


In [19]:
multi_retriever = QueryFusionRetriever(
    retrievers=[index.as_retriever(similarity_top_k=5)],
    similarity_top_k=5,
    mode="reciprocal_rerank",
    num_queries=4,        # LLM generates 3 paraphrases + original
    use_async=False
)

In [20]:
query_engine = RetrieverQueryEngine(retriever=multi_retriever)
response = query_engine.query("What is the HR Policy Manual?")
print(response)

2026-06-07 12:38:46,132 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-07 12:38:47,035 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 12:38:47,266 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 12:38:47,627 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 12:38:47,876 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 12:38:49,042 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The HR Policy Manual at NovaTech Solutions Pvt. Ltd. outlines various policies and guidelines related to employment terms, leave policies, compensation and benefits, performance appraisal cycles, code of conduct including policies on workplace harassment, confidentiality, data security, and disciplinary actions. It covers aspects such as probation period, notice periods, working hours, leave entitlements (annual leave, sick leave, maternity leave, paternity leave, casual leave, bereavement leave), salary structure, performance appraisals, EPF contributions, and policies on workplace behavior and compliance.
